# Interpretative Jury Learning

Adjust **`RunConfig`** below, run **`run_full_pipeline(cfg)`**, then read metrics from **`result`**:

- **Training / validation (per epoch):** `result.training_metrics_df()` — columns include `train_loss`, `train_accuracy`, `val_loss`, `val_accuracy`, `learning_rate`.
- **Last epoch only:** `result.final_training_metrics()` — dict with the same fields.
- **Held-out test splits:** `result.eval_metrics_df()` or `result.eval_accuracy_by_split` — accuracies are fractions in `[0, 1]` (`validation` is the same val set used during training).

Set `cfg.verbose = False` for quiet runs; `cfg.show_progress_bar = True` for tqdm over batches.

**Optional plots** (continent breakdown, calibration, etc.) live in `jury_learning.plots`; see the section at the bottom.


In [ ]:
!git clone "https://github.com/EmiDi3/Interpretative-Jury-Learning.git"
%cd Interpretative-Jury-Learning/

### Google Colab: Moral Machine zip on Google Drive

Optional. Mount Drive and extract `moral_machine.zip` into the Colab runtime (fast local disk). Adjust **`ZIP_PATH`** if your Drive folder layout differs. Skip this block when you already have `moral_machine.db` on local disk or next to the repo.

After extraction, the next cell sets **`cfg.db_path`** from **`MORAL_MACHINE_DB`** (defaulting to `moral_machine.db` if you skip this section).

In [ ]:
import zipfile
from pathlib import Path

# Same layout as: drive/My Drive/Interpretative Jury Learning/moral_machine.zip (cwd is usually /content)
ZIP_PATH = Path("/content/drive/My Drive/Interpretative Jury Learning/moral_machine.zip")
EXTRACT_PATH = Path("/content")

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    pass  # not on Colab

if ZIP_PATH.is_file():
    EXTRACT_PATH.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print(f"Extracted {ZIP_PATH} -> {EXTRACT_PATH}")
    # SQLite path after unzip (change if your archive nests the file in a subfolder)
    MORAL_MACHINE_DB = str(EXTRACT_PATH / "moral_machine.db")
else:
    print(f"No zip at {ZIP_PATH}; next cell uses default cfg.db_path (./moral_machine.db).")

In [ ]:
from jury_learning import RunConfig, run_full_pipeline

cfg = RunConfig()

# Colab: after the Drive zip cell, this picks up MORAL_MACHINE_DB; locally defaults to ./moral_machine.db
cfg.db_path = globals().get("MORAL_MACHINE_DB", "moral_machine.db")
cfg.model_path = "moral_jury_dcn_model.pth"

cfg.sql_subset_size = 100000
cfg.batch_size = 1024
cfg.eval_batch_size = 64

cfg.embed_dim = 128
cfg.hidden_dim = 512
cfg.num_cross_layers = 3
cfg.response_encoder_hidden = 64

cfg.epochs = 50
cfg.lr = 1e-3
cfg.lr_phase2 = 1e-4
cfg.freeze_encoder_epoch_fraction = 2 / 3

cfg.verbose = True
cfg.show_progress_bar = False
cfg.use_wandb = False

cfg.device = "auto"

cfg.run_training_stage = True
cfg.run_evaluation_stage = True

result = run_full_pipeline(cfg)


### Metrics


In [ ]:
# Per-epoch train + validation (same val loader as during training)
result.training_metrics_df()


In [ ]:
# Final epoch summary
result.final_training_metrics()


In [ ]:
# Accuracy on validation + stress-test splits (fractions 0–1)
result.eval_metrics_df()


### Optional evaluation plots

Requires **`pip install matplotlib`**; continent aggregation needs **`pip install pycountry_convert`**.

- **`run_evaluation_plots`** — split bar chart, training curves, calibration diagram, continent + country bars (validation and new-user splits).
- Use **`accuracy_by_country`** + **`continent_metrics_table`** / **`country_metrics_table`** if you want tables without plotting.


In [ ]:
from jury_learning.plots import run_evaluation_plots

run_evaluation_plots(
    cfg,
    result.bundle,
    result.model,
    split_metrics=result.eval_accuracy_by_split,
    history=result.training_history,
)
